In [47]:
import os
import random
import shutil
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import noise

# Validation Flow

In [48]:
def generate_defect_mask(
    height: int, width: int, rng: random.Random, center: tuple = None
) -> np.ndarray:
    """
    Generate a single connected binary defect mask using Perlin noise.

    Multi-octave Perlin noise is evaluated on a grid centered at ``center``,
    then Gaussian-weighted so only one coherent blob region exceeds the
    threshold. The result is an organic, fractal-edged defect region that
    closely resembles real anomaly shapes (scratches, contamination patches).

    Args:
        height: Mask height in pixels.
        width:  Mask width in pixels.
        rng:    Seeded random instance for reproducibility.
        center: (cy, cx) pixel coordinates for the defect center.
                If None, a random point in the central 60% of the image is used.

    Returns:
        Float32 numpy array of shape (height, width) with values in {0.0, 1.0},
        where 1.0 marks defect pixels.
    """
    if center is None:
        cy = rng.uniform(0.2, 0.8) * height
        cx = rng.uniform(0.2, 0.8) * width
    else:
        cy, cx = float(center[0]), float(center[1])

    # Perlin noise parameters
    scale = rng.uniform(2.0, 6.0)       # lower → larger, smoother blobs
    octaves = rng.randint(4, 8)          # more octaves → more fractal detail
    persistence = rng.uniform(0.4, 0.7)  # amplitude falloff per octave
    lacunarity = rng.uniform(1.8, 2.2)   # frequency multiplier per octave
    base = rng.randint(0, 255)           # random seed offset for noise field

    perlin = np.zeros((height, width), dtype=np.float32)
    for y in range(height):
        for x in range(width):
            nx = x / width * scale
            ny = y / height * scale
            perlin[y, x] = noise.pnoise2(
                nx, ny,
                octaves=octaves,
                persistence=persistence,
                lacunarity=lacunarity,
                base=base,
            )

    # Normalize to [0, 1]
    perlin = (perlin - perlin.min()) / (perlin.max() - perlin.min() + 1e-8)

    # Gaussian weight centered at (cy, cx) → single connected blob
    sigma = rng.uniform(0.06, 0.09) * min(height, width)
    yy, xx = np.mgrid[0:height, 0:width]
    weight = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma ** 2))

    combined = perlin * weight

    # Threshold to target defect coverage of 0.01–2% of image area
    target_coverage = rng.uniform(0.0001, 0.02)
    threshold = float(np.percentile(combined, (1.0 - target_coverage) * 100))
    mask = (combined >= threshold).astype(np.float32)
    return mask

In [49]:
def inject_defect(
    image: Image.Image,
    texture_dirs: list,
    rng: random.Random,
) -> tuple:
    """
    Inject a synthetic defect into ``image`` using texture blending.

    The defect center is sampled from foreground pixels only (detected by
    comparing each pixel against the background color estimated from the image
    corners). The generated defect mask is then intersected with the foreground
    mask so no defect pixel can land on the background.

    A random texture patch is blended in within the mask:
        result = image * (1 - mask) + texture * mask

    Args:
        image:        PIL RGB image to corrupt.
        texture_dirs: List of directories from which to sample texture images.
                      Typically the ``train/good`` folders of other MVTec categories.
        rng:          Seeded random instance for reproducibility.

    Returns:
        (defected_image, mask_image) — PIL RGB image with the injected defect and
        a grayscale PIL Image (mode "L") of the binary defect mask (0 or 255).
    """
    w, h = image.size
    img_arr = np.array(image, dtype=np.float32) / 255.0

    # Estimate background color from corners and build a foreground mask
    cs = max(5, min(h, w) // 20)
    corner_pixels = np.concatenate([
        img_arr[:cs, :cs].reshape(-1, 3),
        img_arr[:cs, -cs:].reshape(-1, 3),
        img_arr[-cs:, :cs].reshape(-1, 3),
        img_arr[-cs:, -cs:].reshape(-1, 3),
    ], axis=0)
    bg_color = corner_pixels.mean(axis=0)
    diff = np.linalg.norm(img_arr - bg_color, axis=2)
    fg_mask = (diff > 0.1).astype(np.float32)

    # Pick the defect center from a foreground pixel
    fg_pixels = np.argwhere(fg_mask > 0)
    if len(fg_pixels) > 0:
        cy, cx = fg_pixels[rng.randint(0, len(fg_pixels) - 1)]
    else:
        cy, cx = h // 2, w // 2

    # Sample a texture image from a randomly chosen texture directory
    texture_dir = rng.choice(texture_dirs)
    texture_files = sorted(
        f for f in os.listdir(texture_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    )
    if not texture_files:
        raise ValueError(f"No texture images found in: {texture_dir}")

    texture_path = os.path.join(texture_dir, rng.choice(texture_files))
    texture_arr = np.array(
        Image.open(texture_path).convert("RGB").resize((w, h), Image.BILINEAR),
        dtype=np.float32,
    ) / 255.0

    # Generate blob mask then clip it to the foreground — no defect on background
    mask = generate_defect_mask(h, w, rng, center=(cy, cx))
    mask = mask * fg_mask

    mask_3d = mask[:, :, np.newaxis]
    defected_arr = img_arr * (1.0 - mask_3d) + texture_arr * mask_3d
    defected_arr = np.clip(defected_arr * 255, 0, 255).astype(np.uint8)
    mask_uint8 = (mask * 255).astype(np.uint8)

    return Image.fromarray(defected_arr), Image.fromarray(mask_uint8)

In [50]:
def augment(
    image: Image.Image,
    rng: random.Random,
    mask: Image.Image = None,
    target_size: tuple = (256, 256),
) -> tuple:
    """
    Apply a reproducible augmentation pipeline to ``image`` (and optionally ``mask``).

    Serves two purposes:
      1. Anti-lookup: geometric transforms (flip, rotation, crop) ensure the
         final image differs from any publicly known ground-truth pixel data,
         making it infeasible for miners to match images to known MVTec samples.
      2. Edge simulation: photometric transforms (noise, brightness/contrast, blur)
         mimic the imaging conditions on Raspberry Pi / Jetson edge cameras.

    Geometric transforms are applied identically to both image and mask to
    preserve pixel-level correspondence. Photometric transforms are image-only.
    The final step resizes everything to ``target_size`` (canonical 256×256).

    Args:
        image:       PIL RGB image.
        rng:         Seeded random instance — same seed as the rest of the
                     pipeline so transforms are fully reproducible per round.
        mask:        Optional grayscale PIL mask (mode "L"). Geometric transforms
                     are mirrored exactly using nearest-neighbour resampling so
                     binary mask values are never interpolated.
        target_size: (width, height) of the output. Default 256×256.

    Returns:
        (aug_image, aug_mask) — aug_mask is None when no mask was provided.
    """
    # ---------------------------------------------------------------- geometric
    # 1. Random horizontal flip
    if rng.random() < 0.5:
        image = image.transpose(Image.FLIP_LEFT_RIGHT)
        if mask is not None:
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

    # 2. Random vertical flip
    if rng.random() < 0.5:
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
        if mask is not None:
            mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

    # 3. Random rotation ±15° — bilinear for image, nearest for mask
    angle = rng.uniform(-15, 15)
    image = image.rotate(angle, resample=Image.BILINEAR, expand=False)
    if mask is not None:
        mask = mask.rotate(angle, resample=Image.NEAREST, expand=False)

    # 4. Random crop (80–100% of image) then resize back to original dimensions
    w, h = image.size
    crop_scale = rng.uniform(0.8, 1.0)
    crop_w, crop_h = int(w * crop_scale), int(h * crop_scale)
    x0 = rng.randint(0, w - crop_w)
    y0 = rng.randint(0, h - crop_h)
    box = (x0, y0, x0 + crop_w, y0 + crop_h)
    image = image.crop(box).resize((w, h), Image.BILINEAR)
    if mask is not None:
        mask = mask.crop(box).resize((w, h), Image.NEAREST)

    # 5. Resize to canonical target resolution
    image = image.resize(target_size, Image.BILINEAR)
    if mask is not None:
        mask = mask.resize(target_size, Image.NEAREST)

    # ----------------------------------------------------------- photometric
    # 6. Gaussian sensor noise  σ ∈ [0, 8] / 255
    noise_sigma = rng.uniform(0.0, 8.0) / 255.0
    if noise_sigma > 0:
        np_rng = np.random.RandomState(rng.randint(0, 2**31 - 1))
        img_arr = np.array(image, dtype=np.float32) / 255.0
        img_arr += np_rng.randn(*img_arr.shape).astype(np.float32) * noise_sigma
        image = Image.fromarray(np.clip(img_arr * 255, 0, 255).astype(np.uint8))

    # 7. Brightness jitter ±15%
    image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.85, 1.15))

    # 8. Contrast jitter ±15%
    image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.85, 1.15))

    # 9. Optional mild Gaussian blur (50% chance, radius 0–1.5 px)
    if rng.random() < 0.5:
        image = image.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.0, 1.5)))

    return image, mask

In [ ]:
def generate_validation_dataset(
    src_dataset_path: str,
    output_path: str,
    dtd_path: str,
    num_images: int = 20,
    seed: int = 42,
    p: float = 0.5,
    real_defect_ratio: float = 0.3,
) -> None:
    """
    Generate a hybrid validation dataset from the MVTec_AD bottle category.

    In total ``num_images`` images are produced. With overall probability ``p``
    an image is defective; of those defective images, ``real_defect_ratio``
    come from the real MVTec test split (with their original ground-truth masks)
    and the rest are synthetically generated (Perlin-noise mask + DTD texture
    blend).  Good images are sampled from the train/good split.
    Augmentation is applied to all images as the last step before saving.

    Output layout:
        output_path/
            ground_truth/
                synthetic/              # masks for synthetic defects
                broken_large/           # original MVTec masks (renamed *_mask.png)
                broken_small/
                contamination/
            test/
                good/                   # good images
                synthetic/              # synthetic defective images
                broken_large/           # real defective images
                broken_small/
                contamination/

    Args:
        src_dataset_path:  Path to the MVTec_AD root (contains ``bottle/``).
        output_path:       Destination folder for the generated dataset.
        dtd_path:          Path to the DTD dataset root (contains ``images/``).
        num_images:        Total number of images in the output dataset.
        seed:              Random seed — should be derived from the block hash
                           each validation round so miners cannot memorise the set.
        p:                 Overall probability that any given image is defective.
        real_defect_ratio: Fraction of defective images sourced from the real
                           MVTec test split (0.0 = all synthetic, 1.0 = all real).
    """
    # ------------------------------------------------------------------ paths
    good_train_dir = os.path.join(src_dataset_path, "bottle", "train", "good")
    test_dir = os.path.join(src_dataset_path, "bottle", "test")
    gt_dir = os.path.join(src_dataset_path, "bottle", "ground_truth")
    real_defect_categories = ["broken_large", "broken_small", "contamination"]

    for d in [good_train_dir, test_dir, gt_dir]:
        if not os.path.isdir(d):
            raise FileNotFoundError(f"Required directory not found: {d}")

    dtd_images_dir = os.path.join(dtd_path, "images")
    if not os.path.isdir(dtd_images_dir):
        raise FileNotFoundError(f"DTD images directory not found: {dtd_images_dir}")

    texture_dirs = sorted(
        os.path.join(dtd_images_dir, cat)
        for cat in os.listdir(dtd_images_dir)
        if os.path.isdir(os.path.join(dtd_images_dir, cat))
    )
    if not texture_dirs:
        raise RuntimeError(f"No texture categories found in: {dtd_images_dir}")

    # ----------------------------------------------------------------- wipe output
    # Always start from a clean slate so no stale files from previous runs survive.
    if os.path.exists(output_path):
        shutil.rmtree(output_path)

    # ------------------------------------------- compute split sizes
    rng = random.Random(seed)
    num_defective = round(num_images * p)
    num_good = num_images - num_defective
    num_real = round(num_defective * real_defect_ratio)
    num_synthetic = num_defective - num_real

    # ------------------------------------------- create output dirs
    def make_dirs(*parts):
        path = os.path.join(output_path, *parts)
        os.makedirs(path, exist_ok=True)
        return path

    test_good_dir = make_dirs("test", "good")
    test_syn_dir = make_dirs("test", "synthetic")
    gt_syn_dir = make_dirs("ground_truth", "synthetic")
    real_out_dirs = {
        cat: (make_dirs("test", cat), make_dirs("ground_truth", cat))
        for cat in real_defect_categories
    }

    # ------------------------------------------- good images (train/good)
    train_good_files = sorted(
        f for f in os.listdir(good_train_dir)
        if os.path.isfile(os.path.join(good_train_dir, f))
    )
    selected_good = rng.sample(train_good_files, min(num_good, len(train_good_files)))
    for filename in selected_good:
        src = os.path.join(good_train_dir, filename)
        image = Image.open(src).convert("RGB")
        image, _ = augment(image, rng)
        image.save(os.path.join(test_good_dir, filename))

    # ------------------------------------------- synthetic defective images
    synthetic_pool = [f for f in train_good_files if f not in selected_good]
    selected_synthetic = rng.sample(synthetic_pool, min(num_synthetic, len(synthetic_pool)))
    for filename in selected_synthetic:
        src = os.path.join(good_train_dir, filename)
        image = Image.open(src).convert("RGB")
        defected_image, mask_image = inject_defect(image, texture_dirs, rng)
        defected_image, mask_image = augment(defected_image, rng, mask=mask_image)
        defected_image.save(os.path.join(test_syn_dir, filename))
        mask_image.save(os.path.join(gt_syn_dir, filename))

    # ------------------------------------------- real defective images
    # Gather all real defect samples across categories
    real_pool = []
    for cat in real_defect_categories:
        cat_test_dir = os.path.join(test_dir, cat)
        for f in sorted(os.listdir(cat_test_dir)):
            if os.path.isfile(os.path.join(cat_test_dir, f)):
                real_pool.append((cat, f))

    selected_real = rng.sample(real_pool, min(num_real, len(real_pool)))
    for cat, filename in selected_real:
        stem = os.path.splitext(filename)[0]
        mask_filename = f"{stem}_mask.png"

        src_img = os.path.join(test_dir, cat, filename)
        src_mask = os.path.join(gt_dir, cat, mask_filename)

        image = Image.open(src_img).convert("RGB")
        mask_pil = Image.open(src_mask).convert("L")
        image, mask_pil = augment(image, rng, mask=mask_pil)
        test_cat_dir, gt_cat_dir = real_out_dirs[cat]
        image.save(os.path.join(test_cat_dir, filename))
        mask_pil.save(os.path.join(gt_cat_dir, mask_filename))

    print(
        f"Validation dataset created at '{output_path}':\n"
        f"  {num_good} good images (train split)\n"
        f"  {len(selected_synthetic)} synthetic defective images\n"
        f"  {len(selected_real)} real defective images "
        f"({', '.join(f'{sum(1 for c,_ in selected_real if c==cat)} {cat}' for cat in real_defect_categories)})\n"
        f"  seed={seed}"
    )

In [52]:
generate_validation_dataset(
    src_dataset_path="../data/datasets/MVTec_AD",
    output_path="../data/datasets/validation_dataset",
    dtd_path="../data/datasets/dtd",
    num_images=50,
    seed=42,
    p=0.5,
    real_defect_ratio=0.3,
)

Validation dataset created at '../data/datasets/validation_dataset':
  25 good images (train split)
  17 synthetic defective images
  8 real defective images (3 broken_large, 2 broken_small, 3 contamination)
  seed=42


# Incentive Mechanism